# Evaluate your machine-learning model

Authors: [Alexandre Gramfort](http://alexandre.gramfort.net), [Thomas Moreau](https://tommoral.github.io/about.html), and [Pedro L. C. Rodrigues](https://plcrodrigues.github.io/).

## Table of contents

**Cross-validation**

* [1 The benefits of cross-validation](#benefitscv)
    * [1.1 Load our dataset](#benefitscv_load)
    * [1.2 Empirical error vs generalization error](#benefitscv_empirical)
    * [1.3 A single error is not enough... what about the variance?](#benefitscv_single)
    * [1.4 Effect of the sample size on the variance analysis](#benefitscv_effect)
* [2 Comparing results versus baseline and chance level](#comparing)
* [3 Choice of cross-validation](#choicecv)

**Metrics**

* [4 Regression metrics](#regressionmetrics)
    * [4.1 Mean squared error (MSE)](#mse)
    * [4.2 Mean absolute error (MAE)](#mae)
    * [4.3 Scale free metrics](#scale-free)

* [5 Classification metrics](#classificationmetrics)
    * [5.1 Accuracy loss](#accuracy)
    * [5.2 Precision and recall](#precision-recall)
    * [5.3 F1 metric](#f1-metric)
    * [5.4 ROC/AUC](#roc-auc)
    * [5.5 Multi-class metrics](#multi-class)    

A core question in machine learning is how to evaluate the performance of a model once it's parameters are estimated (i.e. the model has been trained). In this notebook, we aim at presenting how you should answer this question in a statistically sound way. First, we will present the benefits of using cross-validation for this task and then have a quick look at different strategies and metrics that one should use in supervised learning.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1 The benefits of cross-validation<a class="anchor" id="benefitscv"></a> [↑](#Table-of-contents)

## 1.1 Load our dataset<a class="anchor" id="benefitscv_load"></a> [↑](#Table-of-contents)

To illustrate our discussion, we will use the [California housing](https://inria.github.io/scikit-learn-mooc/python_scripts/datasets_california_housing.html) dataset. 

This dataset is a regression problem in which we want to estimate the median housing value given housing information.

In [ ]:
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
y *= 100

In [ ]:
X.head()

In [ ]:
y.head()

## 1.2 Empirical error vs. generalization error<a class="anchor" id="benefitscv_empirical"></a> [↑](#Table-of-contents)

Let's start by creating a regressor which will be a decision tree.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

regressor = DecisionTreeRegressor()

Let's first train our regressor on the full dataset.

In [ ]:
regressor.fit(X, y)

Now that our regressor is trained, we can check the regressor performance. For this purpose, we will use the mean absolute error which will give us an error in the native unit of the target, i.e. k$.

In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = regressor.predict(X)
score = mean_absolute_error(y_pred, y)
print(f"In average, our regressor make an error of {score:.2f} k$")

<div class="alert alert-success">
    <b>QUESTION</b>:
     <ul>
     <li>Are you surprised to get such performance with this regressor?</li>
     <li>Do you expect this regressor to perform this way in production?</li>
     </ul>
</div>

Surprisingly or not, our regressor is perfect. However, we are currently not able to confirm if our model would work on future unseen data. We can simulate this stage by splitting our dataset into two datasets and keep a dataset out of the learning process.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

In [ ]:
?train_test_split

In [ ]:
regressor.fit(X_train, y_train)

In [ ]:
y_pred = regressor.predict(X_train)
score = mean_absolute_error(y_pred, y_train)
print(f"In average, our regressor make an error of {score:.2f} k$")

When evaluating our regressor on the training data, we still get a perfect model. This error is called the **empirical error**. Let see, if we get as lucky on the left-out dataset.

In [ ]:
y_pred = regressor.predict(X_test)
score = mean_absolute_error(y_pred, y_test)
print(f"In average, our regressor make an error of {score:.2f} k$")

... and not. By evaluating our model on a left-out set, we are checking that our model will be able to work on unseen data and thus if it is able to generalize. This type of error is called the **generalization error** and is the one that any data scientist hopes to decrease when creating a model.

<div class="alert alert-success">
    <b>QUESTION</b>:
     <ul>
     <li>Are we sure that our algorithm is robust?</li>
     </ul>
</div>

In [ ]:
for _ in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y)
    regressor.fit(X_train, y_train)
    print(regressor.score(X_test, y_test))

### Generalization error:

The aim of model training is to select the model $f$ out of a class of models $\mathcal F$ that minimizes a measure of the risk. The risk is measured with a loss $l$ between the true value $y$ associated to $x$ and the prediction $f(x)$ and thus we want to find:  

$$f^\star = \arg\min_{f \in \mathcal F}\mathbb E_{(x, y) \sim \pi}[l(f(x), y]$$

The issue is that we cannot compute the expectation $\mathbb E_{(x, y) \sim \pi}$ because we don't know the input distribution $\pi$. Therefore, we approximate it with a set of examples $\{(x_1, y_1), \dots (x_N, y_N)\}$ drawn i.i.d. from $\pi$ and use the Empirical Risk Minimization (ERM):

$$\widehat{f} = \arg\min_{f \in \mathcal F}\frac1N\sum_{i=1}^N l(f(x_i), y_i]$$

If the samples are drawn independently, we know that the error has a variance of $\mathcal O\left(\frac{1}{\sqrt{N}}\right)$. 

Thus there is a gap between the minimizer of the risk and the minimizer of the empirical risk. If we optimize too much for the ERM, the gap might be big and the selected model will have bad performance on unseen data. This is what is called <b>overfitting</b>. To control, this, one need to have a measure of the risk independent from the measure of the risk which is used to select the model: the Empirical Risk on the test set!

## 1.3 A single error is not enough... what about the variance?<a class="anchor" id="benefitscv_single"></a> [↑](#Table-of-contents)

While we were able to estimate the generalization error, we are indeed unable to know anything about the variance of our model and thus if it is robust or not. This is where the framework of cross-validation is used. Indeed, we can repeat our experiment and compute several time our generalization error and get intuition about the stability of our model.

The simplest way that we can think of is to shuffle our data and split it into two sets as we previously did and repeat several time our experiment. In scikit-learn, using the function `cross_validate` with the cross-validation `ShuffleSplit` allows us to make such evaluation.

In [ ]:
%%time
from sklearn.model_selection import cross_validate
from sklearn.model_selection import ShuffleSplit

cv = ShuffleSplit(n_splits=30, test_size=0.2)

result_cv = cross_validate(
    regressor, X, y, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=6
)
result_cv

Once our cross-validation is done, we see that we got a Python dictionary with information regarding the cross-validation. Let's use a pandas dataframe to easily explore the content of the results.

In [ ]:
result_cv = pd.DataFrame(result_cv)
result_cv

We got information about the training and testing time which we can discard for the moment. However, we see that we got the `test_score` which is our generalization scores. We have 30 numbers because we repeated 30 times the same experiment by shuffling the data. Now, we can plot the distribution of these results.

In [ ]:
sns.displot(-result_cv["test_score"], kde=True)
_ = plt.xlabel("Mean absolute error (k$)")

We can observe that the generalization error is centered around 45.5 k\\$ and range from 44 k\\$ to 47 k\\$.

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
     <li>Can we conclude anything about the performance of our regressor?</li>
     <li>Can we conclude anything about our cross-validation analysis?</li>
     </ul>
    <b>Hint: </b>Plot the distribution of the target.
</div>

To be able to know if we can use our cross-validation results, we should put them into perspective with the problem that we try to solve. To know more about this, we can check the distribution of the original target (which we should have done beforehand).

In [ ]:
# %load solutions/solution_1.py

We see that the median value range from 50 k\$ up to 500 k\$. Thus an error range of 3 k\$ means that our cross-validation results can be trusted and do not suffer from an execessive variance. Regarding the performance of our model itself, we can see that making an error of 45 k\$ would be problematic even more if this happen for housing with low value. However, we also see some limitation regarding the metric that we are using. Making an error of 45 k\$ for a target at 50 k\$ and at 500 k\$ should not have the same impact. We should instead use the mean absolute percentage error which will give a relative error.

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
     <li>Check the distribution of the training errors similarly to what we did for the generalization errors.</li>
     <li>What can you conclude?</li>
     </ul>
</div>

What about the empirical error?

In [ ]:
# %load solutions/solution_2.py

In [ ]:
# %load solutions/solution_3.py

## 1.4 Effect of the sample size on the variance analysis<a class="anchor" id="benefitscv_effect"></a> [↑](#Table-of-contents)

We are quite lucky. Our dataset count many samples.

In [ ]:
y.size

Let's make an experiment and reduce the number of samples and repeat the previous experiment.

In [ ]:
def make_cv_analysis(regressor, X, y):
    cv = ShuffleSplit(n_splits=10, test_size=0.2)
    result_cv = pd.DataFrame(
        cross_validate(
            regressor, X, y, cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )
    )
    return y.size, (result_cv["test_score"] * -1).values

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
     <li>Make an experiment by subsampling the dataset and plotting the distribution of the generalization errors.</li>
     <li>What can you conclude?</li>
     </ul>
</div>

In [ ]:
# %load solutions/solution_4.py

We see that with a low number of samples, the variance is much larger. Indeed, for low number of sample, we cannot even trust our cross-validation and therefore cannot conclude anything about our regressor. Therefore, it is really important to make experiment with a large enough sample size to be sure about the conclusions which would be drawn.

# 2 Comparing results versus baseline and chance level<a class="anchor" id="comparing"></a> [↑](#Table-of-contents)

Previously, we compare the generalization error by taking into account the target distribution. A good practice is to compare the generalization error with a dummy baseline and the chance level. In regression, we could use the `DummyRegressor` and predict the mean target without using the data. The chance level can be determined by permuting the labels and check the difference of result.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor()
result_dummy = cross_validate(
    dummy, X, y, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1
)

In [ ]:
from sklearn.model_selection import permutation_test_score

score, permutation_score, pvalue = permutation_test_score(
    regressor, X, y, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1, n_permutations=10,
)

We plot the generalization errors for each of the experiment. We see that even our regressor does not perform well, it is far above chances our a regressor that would predict the mean target.

In [ ]:
final_result = pd.concat(
    [
        result_cv["test_score"] * -1,
        pd.Series(result_dummy["test_score"]) * -1,
        pd.Series(permutation_score) * -1,
    ], axis=1
).rename(columns={
    "test_score": "Cross-validation score",
    0: "Dummy score",
    1: "Permuted score",
})

In [ ]:
sns.displot(final_result, kind="kde")
_ = plt.xlabel("Mean absolute error (k$)")

# 3 Choice of cross-validation<a class="anchor" id="choicecv"></a> [↑](#Table-of-contents)

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
     <li>Is `ShuffleSplit` cross-validation always the best cross-validation to use?</li>
     </ul>
</div>

Let's take an example of some financial quotes. These are the value of compagny  stocks with the time.

In [ ]:
import pandas as pd
import os
from urllib.request import urlretrieve

symbols = {'TOT': 'Total', 'XOM': 'Exxon', 'CVX': 'Chevron',
           'COP': 'ConocoPhillips', 'VLO': 'Valero Energy'}

quotes = pd.DataFrame()

for symbol, name in symbols.items():
    url = ('https://raw.githubusercontent.com/scikit-learn/examples-data/'
           'master/financial-data/{}.csv')
    filename = "data/{}.csv".format(symbol)
    if not os.path.exists(filename):
        urlretrieve(url.format(symbol), filename)
    this_quote = pd.read_csv(filename)
    quotes[name] = this_quote['open']
quotes.index = pd.to_datetime(this_quote['date'])

quotes.drop(["Total"], axis=1, inplace=True)

In [ ]:
_, ax = plt.subplots(figsize=(10, 6))
_ = quotes.plot(ax=ax)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

X, y = quotes.drop(columns=["Chevron"]), quotes["Chevron"]
regressor = GradientBoostingRegressor()

In [ ]:
cv = ShuffleSplit(n_splits=30)
result_cv = pd.DataFrame(
    cross_validate(
        regressor, X, y, cv=cv,
    )
)
print(f'Mean R2: {result_cv["test_score"].mean():.2f}')

<div class="alert alert-success">
    <b>QUESTION</b>:
     It seems that we have the perfect regressor. Does this seem normal to you?
</div>

In [ ]:
# %load solutions/solution_5.py

Let's check the different type of cross-validation that are available in scikit-learn:

https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html#sphx-glr-auto-examples-model-selection-plot-cv-indices-py

# 4 Regression metrics<a class="anchor" id="regressionmetrics"></a> [↑](#Table-of-contents)

## 4.1 Mean Squared Error (MSE)<a class="anchor" id="mse"></a> [↑](#Table-of-contents)

The most classical way to measure distance is to use the the $\ell_2$-norm. The MSE metric simply computes the average $\ell_2$-distance between the prediction and the true target, *i.e.*
$$
    MSE(y, \widehat y) = \frac{1}{N}\sum_{i=1}^N (y_i - \widehat y_i)^2
$$

While this metric can be useful to evaluate linear regression models with gaussian noise, it can fail to distinguish between good and bad cases when the assumptions of the linear model are broken. This is illustrated with the [Anscombe Quartet](https://fr.wikipedia.org/wiki/Quartet_d%27Anscombe), where 4 very different datasets have the same MSE error for linear regression. Note that as the `LinearRegression` fo scikit-learn is an Ordinary Least Squares, the reported MSE is the lowest one that can be achieved for a linear model.


In [ ]:
import json

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
import pandas as pd

reg = LinearRegression()

with open('anscombe_quartet.json') as f:
    anscombe_quartet = np.array(json.load(f))
    
fig, axes = plt.subplots(2, 2, figsize=(10, 10), sharey=True, sharex=True)

for data, ax in zip(anscombe_quartet, axes.ravel()):
    
    x, y = data[0], data[1]
    reg.fit(x[:, None], y)
    y_pred = reg.predict(x[:, None])
    ax.scatter(x, y)
    
    xlim = ax.get_xlim()
    t = np.arange(*xlim)
    ax.plot(t, reg.coef_ * t + reg.intercept_, 'k--')
    ax.set_xlim(xlim)
    
    ax.set_title(f"RMSE = {np.sqrt(mean_squared_error(y_pred, y)):.2e}")


    df = pd.DataFrame(np.array(data).T)
    print(reg.coef_, reg.intercept_)

In [ ]:
import pandas as pd
for i, data in enumerate(anscombe_quartet):
    print(i)
    df = pd.DataFrame(data.T)
    print(df.describe())

This metric can also be rescaled to take into account the variance of the original $y$. 

This is called the Explained Variance:
$$
    \text{Var}_{\text{explained}}(y, \widehat y) = \frac{\sum_{i=1}^N (y_i - \widehat y_i)^2
                                         }{\sum_{i=1}^N (y_i - \bar y)^2}
                                   = \frac{\text{MSE}(y, \widehat y)}{\text{Var}(y)}
    \enspace ,
$$
where $\bar y$ is the mean value of $y$. 

This value is also linked to the [coefficient of determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) as $R^2 = 1 - \text{Var}_{\text{explained}}$.

In [ ]:
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# The Boston house-price data
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

# Shuffle the data
data, target = shuffle(data, target, random_state=0)
regressor = RandomForestRegressor(n_estimators=10)

print("R2 score:", cross_val_score(regressor, data, target, cv=5))

In [ ]:
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# The Boston house-price data
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

# Shuffle the data
data, target = shuffle(data, target, random_state=0)
regressor = RandomForestRegressor(n_estimators=10)

print("R2 score:", cross_val_score(regressor, data, target, cv=5))

In [ ]:
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
raw_df

In [ ]:
data.shape

R2 score is convenient because it has a natural scaling: 1 is
perfect prediction, and 0 is around chance

Now let us see which houses are easier to predict. Feature 3 tells us whether the house is along Charles river or not.

In [ ]:
print("Along Charles River:",
      cross_val_score(regressor, data[data[:, 3] == 1],
                      target[data[:, 3] == 1], cv=5))
print("Not Along Charles River:",
      cross_val_score(regressor, data[data[:, 3] == 0],
                      target[data[:, 3] == 0], cv=5))

Note that despite the name, $R^2$ can be a negative value! See the [docs](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) as well.

In [ ]:
print("Along Charles River:",
      cross_val_score(regressor, data[data[:, 3] == 1],
                      target[data[:, 3] == 1], cv=5,
                      scoring='neg_mean_squared_error'))
print("Not Along Charles River:",
      cross_val_score(regressor, data[data[:, 3] == 0],
                      target[data[:, 3] == 0], cv=5,
                      scoring='neg_mean_squared_error'))

From [this](https://stackoverflow.com/a/72388329) StackOverflow answer:

> When we use the term "loss" it is heavily implied that we want to **minimize** it. However when we talk about "score" it's implied that we want to **maximize** it (e.g. accuracy of a classifier). Therefore, when scoring a regression model, a maximal score would mean a minimal loss, i.e. maximize the negated loss.

## 4.2 Mean Absolute Error (MAE)<a class="anchor" id="mae"></a> [↑](#Table-of-contents)

Another popular metric for regression is to use the $\ell_1$-norm to quantify the difference between the predictions and the target *i.e.*
$$
    MAE(y, \widehat y) = \frac{1}{N}\sum_{i=1}^N |y_i - \widehat y_i| \enspace .
$$
This metric is less sensitive to strong errors than the MSE, as it does not square the error:

In [ ]:
eps = 1.35
t = np.linspace(-3, 3, 1000)
plt.plot(t, abs(t))
plt.plot(t, t*t)

Let's use this MAE metric on the [Anscombe Quartet](https://fr.wikipedia.org/wiki/Quartet_d%27Anscombe)

In [ ]:
from sklearn.metrics import mean_absolute_error
import pandas as pd

    
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for (x, y), ax in zip(anscombe_quartet, axes.ravel()):
    reg.fit(x[:, None], y)
    y_pred = reg.predict(x[:, None])
    ax.scatter(x, y)
    
    xlim = ax.get_xlim()
    t = np.arange(*xlim)
    ax.plot(t, reg.coef_ * t + reg.intercept_, 'k--')
    ax.set_xlim(xlim)
    ax.set_title(f"MAE = {mean_absolute_error(y_pred, y):.2e}")

In particular, the MAE metric is less sensitive to outliers.

But we're in a rather **strange** situation: the model is trained to minimize one objective function but we evaluate its performance based on another metric!

Let's see if we can find a regressor model that aims to optimize something closer
to the MAE. 

For this we will use the [HuberRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.HuberRegressor.html)

In [ ]:
from sklearn.linear_model import HuberRegressor

reg = HuberRegressor()
    
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for (x, y), ax in zip(anscombe_quartet, axes.ravel()):
    reg.fit(x[:, None], y)
    y_pred = reg.predict(x[:, None])
    ax.scatter(x, y)
    
    xlim = ax.get_xlim()
    t = np.arange(*xlim)
    ax.plot(t, reg.coef_ * t + reg.intercept_, 'k--')
    ax.set_xlim(xlim)
    ax.set_title(f"MAE = {mean_absolute_error(y_pred, y):.2e}")

Another benefit from this metric is that it permits to report an error in the correct unit, meaningful for an application. 

Taking back the boston dataset:

In [ ]:
data, target = shuffle(data, target, random_state=0)

print(cross_val_score(regressor, data, target, cv=5,
                      scoring='neg_mean_absolute_error'))

## 4.3 Scale free metrics<a class="anchor" id="scale-free"></a> [↑](#Table-of-contents)
## 

Depending on the application of the regression problem, it might be necessary to use a metric which quantifies the relative error instead of the absolute error. There exists various ways to define such a metric, in particular:

* The Mean Absolute Pourcentage Error (MAPE): $\displaystyle \frac{1}{N}\sum_{i=1}^N \left| \frac{y_i-\widehat y_i}{y_i}\right|~.$ This computes the error between the predicted value and the target, and rescales it compared to the value of the target $y$. This means that the same error of 1 won't have the same effect depending on the fact that $y=1$ or $y=1000$, which can be useful in various situation. It is [available in scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html) starting from version 0.24.

> [PLCR] Note that the MAPE will punish harder the over-predictions than the under-predictions. For instance, if the estimate is 10x the true value the MAPE is 9 whereas if it is 0.1x the MAPE is 0.9. In other words, the estimator will have a bias towards under-predictions, which might not be desirable. 

> **Question**: How would you train a linear regressor to minimize this score?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

X, y = data, target
est = LinearRegression()

cv = KFold(n_splits=5, shuffle=True)

print('usual training')
mse_list = []
mape_list = []
for train_idx, test_idx in cv.split(X, y):
    est.fit(X[train_idx], y[train_idx])
    y_true = y[test_idx]
    y_pred = est.predict(X[test_idx])
    mse_list.append(mean_squared_error(y_true, y_pred))
    mape_list.append(mean_absolute_percentage_error(y_true, y_pred))
print('MSE:', np.mean(mse_list))
print('MAPE:', np.mean(mape_list))

print('')

print('weighted training')
mse_list = []
mape_list = []
for train_idx, test_idx in cv.split(X, y):
    est.fit(X[train_idx], y[train_idx], sample_weight=1.0/np.abs(y[train_idx]))
    y_true = y[test_idx]
    y_pred = est.predict(X[test_idx])
    mse_list.append(mean_squared_error(y_true, y_pred))
    mape_list.append(mean_absolute_percentage_error(y_true, y_pred))
print('MSE:', np.mean(mse_list))
print('MAPE:', np.mean(mape_list))

But what if we had a pipeline? Try it: the cell below **is expected to fail**. A plain `Pipeline` does not know which step should receive `sample_weight`, so it refuses the argument. The next cell shows the fix.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = make_pipeline(StandardScaler(), LinearRegression())

print('weighted training')
mse_list = []
mape_list = []
for train_idx, test_idx in cv.split(X, y):
    pipeline.fit(X[train_idx], y[train_idx], sample_weight=1.0/np.abs(y[train_idx]))
    y_true = y[test_idx]
    y_pred = pipeline.predict(X[test_idx])
    mse_list.append(mean_squared_error(y_true, y_pred))
    mape_list.append(mean_absolute_percentage_error(y_true, y_pred))
print('MSE:', np.mean(mse_list))
print('MAPE:', np.mean(mape_list))

Scikit-learn has recently launched a new feature called **metadata routing** that facilitates the above procedure:

In [ ]:
import sklearn
sklearn.set_config(enable_metadata_routing=True)

scaler = StandardScaler().set_fit_request(sample_weight=False)
reg = LinearRegression().set_fit_request(sample_weight=True)

pipeline = make_pipeline(scaler, reg)

print('weighted training')
mse_list = []
mape_list = []
for train_idx, test_idx in cv.split(X, y):
    pipeline.fit(X[train_idx], y[train_idx], sample_weight=1.0/np.abs(y[train_idx]))
    y_true = y[test_idx]
    y_pred = pipeline.predict(X[test_idx])
    mse_list.append(mean_squared_error(y_true, y_pred))
    mape_list.append(mean_absolute_percentage_error(y_true, y_pred))
print('MSE:', np.mean(mse_list))
print('MAPE:', np.mean(mape_list))

* The Mean Squared Log Error (MSLE): $\displaystyle \frac{1}{N}\sum_{i=1}^N \Big(\log(1 + y_i) - \log(1 + \widehat y_i)\Big)^2$. This computes a risk metric corresponding to the expected value of the squared logarithmic (quadratic) error or loss. This metric is best to use when targets have exponential growth, such as population counts, average sales of a commodity over a span of years etc. Note that this metric penalizes an under-predicted estimate more than an over-predicted estimate.

> [PLCR] Consider a regression problem where we have to **predict the time taken** by an agent to deliver the food to customers (e.g. Deliveroo). Now, if the regression model which we built overestimates the delivery time, the delivery agent then gets a relaxation on the time he takes to deliver food and this small overestimation is acceptable. But the problem arises when the predicted delivery time is less than the actual trip takes, in this case, the delivery agent is more likely to miss the deadline, as a result, the customer reviews can be affected.

> **Question**: How would you train a linear regressor to minimize this loss?

In [ ]:
import numpy as np 
import scipy.stats as stats
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5), ncols=2)
stats.probplot(y, dist="norm", plot=ax[0])
stats.probplot(np.log(1+ y), dist="norm", plot=ax[1])
plt.show()

# 5 Classification metrics<a class="anchor" id="classificationmetrics"></a> [↑](#Table-of-contents)

In binary classification we have two classes $\{0, 1\}$, often called Positive and Negative and we try to predict the class $y_i$ for each sample $X_i$. A simple example would be to try to predict whether a dog is present or not in a given image. In this case, the Positive class is the class of all photos of dogs and the Negative class includes all the other photos. The goal of the classifier is to predict, for each photo, if it is Positive (P) or Negative (N): is there a dog in the photo?

An interesting way to visualize the performances of the classifier is to look at the confusion matrix of the classifier.

<img src="images/confusion_dog.png" width="450"/>

This confusion matrix displays all the information about the performance of our classifier.
We can see for each class whether or not it is well classified. In this example, 2 photos with dogs were classified as Negative (no dog!), and 1 photo without a dog was classified as Positive (dog!).

- When a Positive sample is falsely classified as Negative, we call this a False Negative (FN). 
- When a Negative sample is falsely classified as a Positive, it is called a False Positive (FP).

Classification metrics are commonly derived from the confusion matrix. We now present some of these metrics.

See also https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics

In [ ]:
# Create a classification problem and train a classifier to solve it.
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

data, target = make_classification(
    n_samples=400, n_features=20, n_classes=2,
    weights=[.7, .3], random_state=42, flip_y=.2,
)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, random_state=0
)
clf = LogisticRegression(solver='lbfgs')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(clf.score(X_test, y_test))

In [ ]:
# Display the confusion matrix for our classifier
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    columns=['N', 'P'],
    index=['N', 'P']
)
sns.heatmap(cm_df, annot=True,
            cmap='Oranges',)
_ = plt.ylim(2, 0)

If you have recent scikit-learn you can also use the [`ConfusionMatrixDisplay`](https://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html) API:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

## 5.1 Accuracy / Zero-One Loss<a class="anchor" id="accuracy"></a> [↑](#Table-of-contents)

The idea of the accuracy is to measure how well your model predicted the class of the samples. This corresponds to the empirical risk of the model
$$
    \text{Accuracy} = \frac{1}{N}\sum_{i=1}^N 1\{\widehat y_i = y_i\} \enspace,
    \quad \text{Error} = 1 - \text{Accuracy} = \frac{1}{N}\sum_{i=1}^N 1\{\widehat y_i \neq y_i\} \enspace.
$$
It corresponds to the probability of correct classification using the classifier.

This is the most common metric used to evaluate a classification task.  

Given a confusion matrix $C$, the accuracy is computed as $ \dfrac{\text{Tr}(C)}{N} $

In [ ]:
print("Accuracy: ", (y_pred == y_test).mean())
print("Score scikit-learn: ", clf.score(X_test, y_test))

Note that accuracy will not be 0.5 at chance level for any classification problem. It will only be the case if the dataset is balanced. The scikit-learn [DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) is good way to evaluate what is chance level for your problem.

In [ ]:
from sklearn.dummy import DummyClassifier

clf = DummyClassifier(strategy='most_frequent')
clf.fit(X_train, y_train)
print("Score scikit-learn: ", clf.score(X_test, y_test))

To take into account the class imbalance you can use the **balanced accuracy** as score.

The balanced accuracy takes into account the proportions of each class.

See https://scikit-learn.org/stable/modules/model_evaluation.html#balanced-accuracy-score for the definition.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

print("Score scikit-learn: ", balanced_accuracy_score(y_test, clf.predict(X_test)))

## 5.2 Precision and Recall<a class="anchor" id="precision-recall"></a> [↑](#Table-of-contents)

The precision and recall are both defined relatively to a chosen positive class.

- **Precision** is the fraction of positive instances among the instances predicted as positive 
- **Recall** is the fraction of the total amount of positive instances that were actually retrieved.

These two metrics are computed as
$$
        \text{Precision} = \frac{TP}{TP+FP}
        \quad \text{and} \quad
        \text{Recall} = \frac{TP}{TP+FN} \enspace.
$$

<img src="images/precision_recall.png" width="750"/>

While recall is also known as `sensitivity`, the `specificity` (a.k.a. true negative rate) is defined as:

$$
        \text{Specificity} = \frac{TN}{TN+FP} \enspace.
$$

<img src="images/covid.png" width="700">

Taking our example back, with positive class `1`, we get

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
print(f"Precision : {precision}")
print(f"Recall : {recall}")

These two metrics better describe the performance of the classifier, which tends to predict a majority of positive value. We can see in this example that the recall -- *i.e.* the recovery rate of the positive class -- is only at 60% while we have a precision of 80%. This means that our model is conservative. the few examples classified as positive are well chosen but to reach this good precision, we drop a lot of the other positive samples.

It is usefull to note that **these two metrics are complementary**. They result from a tradeoff between:
- Predictions are trustworthy (when the model predicts positive, it is confident that it will be true) 
- You don't reject too many samples (when a positive sample is presented to the model, it will be predicted as positive). 

To illustrate this concept, let's consider the limit case:

* If a model always predicts samples as negative, it has a precision of 1 but it has a recall of 0 as it never finds the positive samples.
* Inversely, if a model always predicts samples as positive, it has a recall of 1 (all the positive samples are classified as positive) but it as a poor precision as all negative samples are also classified as positive. 

## 5.3 F1 metric<a class="anchor" id="f1-metric"></a> [↑](#Table-of-contents)

The inherent issue with precision and recall is that you can't optimize for both at the same time. As they result of a tradeoff, when you maximize one of the metric, the other will automatically decrease. A convenient way to balance these two metrics is to use the F1-score. This metric combines both the precision and recall and tries to balance both. This score is computed as the harmonic mean of the precision $P$ and recall $R$ *i.e.*

$$
    \text{F1} = 2 * \frac{P \times R}{P + R}\enspace.
$$
The use of a harmonic mean permits to give more importance to low scores. Thus, a precision and recall of $(0.8, 0.8)$ will have a larger F1-score compared to $(1, 0.6)$, unlike with arithmetic mean.

In [ ]:
from sklearn.metrics import f1_score
print(f"F1-score : {f1_score(y_test, y_pred, pos_label=1)}")

This metric is often used for model selection, to ensure a **good balance between the precision and recall** in the final model.

The classic F1-score gives the same weight to the precision and the recall. But it is also possible to **change balance between precision and recall** by using a modified f1-score for a given value $\beta > 0$
$$
    \text{F1-score}_\beta =  (1 + \beta^2) * \frac{P . R}{\beta^2 P + R} \enspace .
$$
Setting $\beta = 2$ gives twice as much importance to the recall as to the precision while $\beta = 1/2$ will do the opposite. You can use the slider below to see how the F1-score varies with $\beta$, as a function of precision and recall.

In [ ]:
import numpy as np
t = np.linspace(1e-16, 1, 101)
P, R = np.meshgrid(t, t)

def plot_f1(beta):
    F1 = (1 + beta*beta) * (P*R) / (beta*beta*P+R)
    plt.contourf(P, R, F1, 25, cmap='viridis')
    plt.xlabel('Precision')
    plt.ylabel('Recall')
    plt.clim([0, 1])
    plt.colorbar()

from ipywidgets import interact, widgets

_ = interact(plot_f1, beta=widgets.FloatLogSlider(
    value=1, min=-1, max=1, step=0.05,
    description='$\\beta$')
)

## 5.4 ROC / AUC<a class="anchor" id="roc-auc"></a> [↑](#Table-of-contents)

We take back our `LogisticRegression` classifier:

In [ ]:
clf = LogisticRegression(solver='lbfgs')
clf.fit(X_train, y_train)

Another way to look at the results from our classifier is the ROC curve (Receiver Operating Characteristic). 

Quoting Wikipedia :

> A receiver operating characteristic (ROC), or simply ROC curve, is a graphical plot which illustrates the performance of a binary classifier system as **its discrimination threshold is varied**. It is created by plotting the fraction of true positives out of the positives (TPR = true positive rate) vs. the fraction of false positives out of the negatives (FPR = false positive rate), at various threshold settings. TPR is also known as sensitivity, and FPR is one minus the specificity or true negative rate.

In other words, if the classifier provides a decision function that can be thresholded to control false positives versus false negatives, the ROC curve summarises the different tradeoffs that can be achieved by varying this threshold.

The computation of this curve requires the true labels $y_i$ and the target scores $p_i$, which can either be probability estimates of the positive class, confidence values, or binary decisions. The curve represents the True Positive Rate as a function of the False Positive Rate when varying the decision threshold. In other words, for a decision threshold $\alpha \in [0, 1]$, the ROC curve compute the FPR and TPR of a classifier such that $f(X_i) = 1\{p_i > \alpha)$. A perfect classifier, for which there exist a value $\alpha$ such that all negative samples have a score lower than $\alpha$ and all positive samples have a score higher, would have a ROC curve which would be a step function. A random classifier would have a [diagonal ROC](https://datascience.stackexchange.com/a/31877) curve.

With our example, the ROC curve gives:

In [ ]:
from sklearn.metrics import roc_curve

clf = LogisticRegression(solver='lbfgs')
clf.fit(X_train, y_train)

clfd = DummyClassifier()
clfd.fit(X_train, y_train)

p_test = clf.predict_proba(X_test)[:, 1]
p_testd = clfd.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, clf.predict_proba(X_test)[:, 1], pos_label=1)
plt.plot(fpr, tpr, label='ROC-curve')

fprd, tprd, thresholdsd = roc_curve(y_test, clfd.predict_proba(X_test)[:, 1], pos_label=1)
plt.plot(fprd, tprd, label='Dummy')

plt.plot([0, 0, 1], [0, 1, 1], label='perfect ROC-curve')
plt.plot(range(0, 2), 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.xlim(-0.01, 1)
plt.ylim(0, 1.01)
plt.legend(ncol=2, bbox_to_anchor=(0, 1.05, 1., .3),
           loc=3, borderaxespad=0);

print(thresholds)
print(tpr)

While this curve is convenient to visualize the performance of the classifier, it is usually summarized by the AUC metric which is the area under the ROC curve. This metric scales in $[0, 1]$.

In [ ]:
from sklearn.metrics import roc_auc_score

p_test = clf.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, p_test, pos_label=1)
plt.plot(fpr, tpr, label='ROC-curve')
plt.plot(range(0, 2), 'k--')
plt.fill_between(fpr, tpr, hatch='\\', alpha=0.3, label='AUC')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.xlim(-0.01, 1)
plt.ylim(0, 1.01)
plt.legend(ncol=2, bbox_to_anchor=(0, 1.05, 1., .3),
           loc=3, borderaxespad=0)

auc = roc_auc_score(y_test, p_test)
print(f"AUC : {auc}")

Note that you can do similar curves with precision and recall:

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

p_test = clf.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_test, clf.predict_proba(X_test)[:, 1], pos_label=1
)
plt.plot(recall, precision, label='PR-curve')
plt.plot([0, 1, 1], [1, 1, 0], label='perfect PR-curve')
plt.plot([0, 1], [1, 0], 'k--')
plt.fill_between(recall, precision, hatch='\\', alpha=0.3, label='AUC')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.xlim(-0.01, 1)
plt.ylim(0, 1.01)
plt.legend(ncol=2, bbox_to_anchor=(0, 1.05, 1., .3),
           loc=3, borderaxespad=0)

pr_auc = auc(recall, precision)
print(f"PR-AUC : {pr_auc}")

## 5.5 Multi-class metrics<a class="anchor" id="multi-class"></a> [↑](#Table-of-contents)

Some metrics are essentially defined for binary classification tasks (e.g. F1-score, AUC). In extending a binary metric to multiclass or multilabel problems, the data is treated as a collection of binary problems, one for each class. There are several ways to average binary metric calculations across the set of classes, each of which may be useful in some scenario. Where available, you should select among these using the average parameter.

 * **Macro-average**: simply calculates the mean of the binary metrics, giving equal weight to each class. In problems where infrequent classes are nonetheless important, macro-averaging may be a means of highlighting their performance. On the other hand, the assumption that all classes are equally important is often untrue, such that macro-averaging will over-emphasize the typically low performance on an infrequent class.
 * **Weighted-average**: accounts for class imbalance by computing the average of binary metrics in which each class’s score is weighted by its presence in the true data sample.
 * **Micro-average**: gives each sample-class pair an equal contribution to the overall metric (except as a result of sample-weight). Rather than summing the metric per class, this sums the dividends and divisors that make up the per-class metrics to calculate an overall quotient. Micro-averaging may be preferred in multilabel settings, including multiclass classification where a majority class is to be ignored.

We will see in the second part of the class how imbalanced classes can be handled efficiently.

In [ ]:
# [PLCR]

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

w = np.array([.7, .1, .2])
data, target = make_classification(
    n_samples=600, n_features=20, n_classes=3, n_clusters_per_class=1,
    weights=w, random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, random_state=0
)
clf = LogisticRegression(solver='lbfgs')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f1_score(y_test, y_pred, average=None))
print(np.mean(f1_score(y_test, y_pred, average=None)))
print(f1_score(y_test, y_pred, average='macro'))
print(np.sum(np.multiply(f1_score(y_test, y_pred, average=None), w)))
print(f1_score(y_test, y_pred, average='weighted'))